# Experiment 3 — YOLO12n + Lightweight Local Feature Refinement (LFR)

Experiment 1 (P2) and Experiment 2 (LCSA) were rejected against the frozen YOLO12n baseline.

Experiment 3 returns to the original YOLO12n and tests one different hypothesis:

> improve local texture/detail extraction inside the existing P3 feature path.

Only one modification is made:

```text
P3 → DWConv 3×3 → Conv 1×1 → Detect
```

P4 and P5 remain unchanged.

Controls:
- clean dataset
- 640×640
- batch 16
- 100 epochs
- seed 42
- GPU 0
- same pretrained YOLO12n initialization
- same evaluation protocol


In [1]:
# CELL 1 — Imports

from pathlib import Path
import json
import time
import platform
import copy

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt

from ultralytics import YOLO
import ultralytics

print("=" * 90)
print("EXPERIMENT 3 — YOLO12n + LFR")
print("=" * 90)
print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("Ultralytics :", ultralytics.__version__)
print("CUDA        :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count   :", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}       :", torch.cuda.get_device_name(i))


EXPERIMENT 3 — YOLO12n + LFR
Python      : 3.13.5
PyTorch     : 2.11.0+cu128
Ultralytics : 8.4.116
CUDA        : True
GPU count   : 2
GPU 0       : NVIDIA RTX A4000
GPU 1       : NVIDIA RTX A4000


In [25]:
# CELL 2 — Configuration and safe pretrained-weight discovery

DATASET_ROOT = Path(r"G:\Thesis_Hasan\Dataset\clean_dataset")
DATASET_YAML = DATASET_ROOT / "data.yaml"

BASELINE_ROOT = Path(r"G:\Thesis_Hasan\Models\Baseline\YOLO12n_original")
BASELINE_CONFIG = BASELINE_ROOT / "experiment_config.json"

EXPERIMENT_ROOT = Path(
    r"G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean"
)

for p in [
    EXPERIMENT_ROOT,
    EXPERIMENT_ROOT / "train",
    EXPERIMENT_ROOT / "validation",
    EXPERIMENT_ROOT / "test_evaluation",
    EXPERIMENT_ROOT / "inference",
]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = 0
IMG_SIZE = 640
BATCH = 16
EPOCHS = 100
SEED = 42
PATIENCE = 30

CLASS_NAMES = [
    "Cercospora leaf spot",
    "Flea-Beetles",
    "Healthy leaf",
    "Phomopsis Blight",
    "Phytophthora Blight",
    "Powdery Mildew",
    "Tobacco Mosaic Virus",
]

NC = len(CLASS_NAMES)

if not DATASET_YAML.exists():
    raise FileNotFoundError(DATASET_YAML)

candidates = [
    BASELINE_ROOT / "yolo12n.pt",
    Path(r"G:\Thesis_Hasan\Models\Baseline\YOLOv12n_original\yolo12n.pt"),
    Path(r"G:\Thesis_Hasan\Models\Baseline\YOLOv12n_augmented\yolo12n.pt"),
]

search_root = Path(r"G:\Thesis_Hasan\Models")
if search_root.exists():
    candidates.extend(search_root.rglob("yolo12n.pt"))

seen = set()
local_weights = []
for p in candidates:
    p = Path(p)
    k = str(p).lower()
    if k not in seen and p.exists() and p.is_file():
        seen.add(k)
        local_weights.append(p)

BASE_WEIGHTS = local_weights[0] if local_weights else "yolo12n.pt"

print("=" * 90)
print("EXPERIMENT 3 CONFIGURATION")
print("=" * 90)
print("Dataset YAML :", DATASET_YAML)
print("Base weights :", BASE_WEIGHTS)
print("Experiment   :", EXPERIMENT_ROOT)
print("Epochs       :", EPOCHS)
print("Batch        :", BATCH)
print("Image size   :", IMG_SIZE)
print("Seed         :", SEED)
print("Patience     :", PATIENCE)
print("GPU          :", DEVICE)
print("=" * 90)


EXPERIMENT 3 CONFIGURATION
Dataset YAML : G:\Thesis_Hasan\Dataset\clean_dataset\data.yaml
Base weights : G:\Thesis_Hasan\Models\Baseline\YOLOv12n_original\yolo12n.pt
Experiment   : G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean
Epochs       : 100
Batch        : 16
Image size   : 640
Seed         : 42
Patience     : 30
GPU          : 0


In [3]:
# CELL 3 — Verify dataset YAML and controls

with open(DATASET_YAML, "r", encoding="utf-8") as f:
    dataset_cfg = yaml.safe_load(f)

for key in ["train", "val"]:
    if key not in dataset_cfg:
        raise RuntimeError(f"Dataset YAML missing required key: {key}")

print("train:", dataset_cfg.get("train"))
print("val  :", dataset_cfg.get("val"))
print("test :", dataset_cfg.get("test"))
print()
print("epochs =", EPOCHS)
print("batch  =", BATCH)
print("imgsz  =", IMG_SIZE)
print("seed   =", SEED)
print("device =", DEVICE)


train: train/images
val  : valid/images
test : test/images

epochs = 100
batch  = 16
imgsz  = 640
seed   = 42
device = 0


In [4]:
# CELL 4 — Locate official installed YOLO12 YAML

ULTRALYTICS_ROOT = Path(ultralytics.__file__).resolve().parent
yaml_candidates = list(ULTRALYTICS_ROOT.rglob("yolo12.yaml"))

if not yaml_candidates:
    raise FileNotFoundError(
        f"Installed yolo12.yaml not found under {ULTRALYTICS_ROOT}"
    )

OFFICIAL_YOLO12_YAML = yaml_candidates[0]

with open(OFFICIAL_YOLO12_YAML, "r", encoding="utf-8") as f:
    official_cfg = yaml.safe_load(f)

print("Official YOLO12 YAML:", OFFICIAL_YOLO12_YAML)
print("Backbone layers:", len(official_cfg["backbone"]))
print("Head layers:", len(official_cfg["head"]))


Official YOLO12 YAML: G:\Thesis_Hasan\.venv1\Lib\site-packages\ultralytics\cfg\models\12\yolo12.yaml
Backbone layers: 9
Head layers: 13


In [5]:
# CELL 5 — Build YOLO12n-LFR architecture

cfg = copy.deepcopy(official_cfg)
cfg["nc"] = NC

detect_positions = []
for i, layer in enumerate(cfg["head"]):
    if isinstance(layer, list) and len(layer) >= 3 and str(layer[2]) == "Detect":
        detect_positions.append(i)

if len(detect_positions) != 1:
    raise RuntimeError(
        f"Expected exactly one Detect layer, found {len(detect_positions)}"
    )

detect_pos = detect_positions[0]
detect_inputs = list(cfg["head"][detect_pos][0])

if len(detect_inputs) != 3:
    raise RuntimeError(
        f"Expected P3/P4/P5 Detect inputs, got {detect_inputs}"
    )

P3_INDEX = int(detect_inputs[0])
P4_INDEX = int(detect_inputs[1])
P5_INDEX = int(detect_inputs[2])

cfg["head"] = copy.deepcopy(cfg["head"][:detect_pos])

# YOLO12n P3 is 64 channels.
# Add a lightweight local refinement:
# DWConv 64->64, 3x3
# Conv   64->64, 1x1

lfr_dw_index = len(cfg["backbone"]) + len(cfg["head"])

cfg["head"].append([
    P3_INDEX,
    1,
    "DWConv",
    [64, 3, 1]
])

lfr_pw_index = len(cfg["backbone"]) + len(cfg["head"])

cfg["head"].append([
    lfr_dw_index,
    1,
    "Conv",
    [64, 1, 1]
])

cfg["head"].append([
    [lfr_pw_index, P4_INDEX, P5_INDEX],
    1,
    "Detect",
    [NC]
])

MODEL_YAML = EXPERIMENT_ROOT / "yolo12n_lfr.yaml"

with open(MODEL_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("=" * 90)
print("LFR ARCHITECTURE")
print("=" * 90)
print("Original P3:", P3_INDEX)
print("DWConv     :", lfr_dw_index)
print("Pointwise  :", lfr_pw_index)
print("P4         :", P4_INDEX)
print("P5         :", P5_INDEX)
print("New Detect :", [lfr_pw_index, P4_INDEX, P5_INDEX])
print("Saved      :", MODEL_YAML)


LFR ARCHITECTURE
Original P3: 14
DWConv     : 21
Pointwise  : 22
P4         : 17
P5         : 20
New Detect : [22, 17, 20]
Saved      : G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean\yolo12n_lfr.yaml


In [6]:
# CELL 6 — Build, transfer weights, and verify forward

model = YOLO(str(MODEL_YAML))

print("✓ YOLO12n-LFR parsed.")
model.info(detailed=False, verbose=True)

print("Loading pretrained YOLO12n weights:", BASE_WEIGHTS)
model.load(str(BASE_WEIGHTS))
print("✓ Matching pretrained weights loaded.")

device_obj = torch.device(
    f"cuda:{DEVICE}" if torch.cuda.is_available() else "cpu"
)
model.model.to(device_obj)

dummy = torch.zeros(
    1, 3, IMG_SIZE, IMG_SIZE, device=device_obj
)

with torch.no_grad():
    _ = model.model(dummy)

print("✓ Dummy forward passed.")

detect_layers = [
    m for m in model.model.modules()
    if m.__class__.__name__ == "Detect"
]

if len(detect_layers) != 1:
    raise RuntimeError("Expected exactly one Detect module.")

total_params = sum(
    p.numel() for p in model.model.parameters()
)

print("Detect modules:", len(detect_layers))
print("Total parameters:", total_params)
print("✓ Model architecture verified.")


✓ YOLO12n-LFR parsed.
YOLO12n_lfr summary: 276 layers, 2,505,173 parameters, 2,505,157 gradients, 7.0 GFLOPs
Loading pretrained YOLO12n weights: G:\Thesis_Hasan\Models\Baseline\YOLOv12n_original\yolo12n.pt
Transferred 570/703 items from pretrained weights
✓ Matching pretrained weights loaded.
✓ Dummy forward passed.
Detect modules: 1
Total parameters: 2505173
✓ Model architecture verified.


## Why this modification is controlled

The baseline detector already has P3/P4/P5. LFR does not introduce P2, does not add attention, and does not replace the backbone.

Only the existing P3 feature is locally refined with standard Ultralytics modules. This also avoids the custom-class checkpoint pickling problem encountered during Experiment 2.


In [7]:
# CELL 7 — Training

print("=" * 90)
print("TRAINING YOLO12n-LFR — CLEAN / 640 / BATCH 16")
print("=" * 90)

train_kwargs = {
    "data": str(DATASET_YAML),
    "epochs": EPOCHS,
    "imgsz": IMG_SIZE,
    "batch": BATCH,
    "device": DEVICE,
    "seed": SEED,
    "patience": PATIENCE,
    "project": str(EXPERIMENT_ROOT),
    "name": "train",
    "exist_ok": True,
    "plots": True,
    "verbose": True,
}

# Reuse only the baseline optimizer choice when available.
if BASELINE_CONFIG.exists():
    try:
        with open(BASELINE_CONFIG, "r", encoding="utf-8") as f:
            baseline_cfg_runtime = json.load(f)
        if baseline_cfg_runtime.get("optimizer") is not None:
            train_kwargs["optimizer"] = baseline_cfg_runtime["optimizer"]
    except Exception as e:
        print("Could not read baseline optimizer setting:", repr(e))

train_start = time.time()

results = model.train(**train_kwargs)

elapsed = time.time() - train_start

print()
print("=" * 90)
print("TRAINING FINISHED")
print("=" * 90)
print(f"Training time: {elapsed / 3600:.2f} hours")
print("Best weights:", EXPERIMENT_ROOT / "train" / "weights" / "best.pt")


TRAINING YOLO12n-LFR — CLEAN / 640 / BATCH 16
New https://pypi.org/project/ultralytics/8.4.120 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.116  Python-3.13.5 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=G:\Thesis_Hasan\Dataset\clean_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None

# Resume training after power interruption

Run Cell 8 only if the training run is interrupted.


In [8]:
# CELL 8 — Resume only if needed

LAST_PT = EXPERIMENT_ROOT / "train" / "weights" / "last.pt"

if not LAST_PT.exists():
    raise FileNotFoundError(LAST_PT)

print("Resuming from:", LAST_PT)

resume_model = YOLO(str(LAST_PT))
resume_model.train(
    resume=True,
    device=DEVICE
)

print("✓ Resume finished.")


Resuming from: G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean\train\weights\last.pt
New https://pypi.org/project/ultralytics/8.4.120 available  Update with 'pip install -U ultralytics'
WARNING model 'G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean\train\weights\last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.116  Python-3.13.5 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=Fal

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [9]:
# CELL 9 — Verify checkpoints

BEST_PT = EXPERIMENT_ROOT / "train" / "weights" / "best.pt"
LAST_PT = EXPERIMENT_ROOT / "train" / "weights" / "last.pt"

print("best.pt:", BEST_PT.exists())
print("last.pt:", LAST_PT.exists())

if not BEST_PT.exists():
    raise FileNotFoundError(BEST_PT)


best.pt: True
last.pt: True


In [1]:

# ============================================================
# CUDA SANITY CHECK — RUN AFTER KERNEL RESTART
# ============================================================

import torch

print("=" * 90)
print("CUDA SANITY CHECK")
print("=" * 90)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))

    try:
        torch.cuda.empty_cache()

        x = torch.tensor(
            [1.0, 2.0, 3.0],
            device="cuda:0"
        )

        y = x * 2

        torch.cuda.synchronize()

        print("✓ CUDA tensor operation passed.")
        print("Result:", y.cpu().numpy())

    except Exception as e:

        print("CUDA TEST FAILED:")
        print(repr(e))

else:

    raise RuntimeError(
        "CUDA is not available."
    )

print("=" * 90)

CUDA SANITY CHECK
PyTorch: 2.11.0+cu128
CUDA available: True
GPU count: 2
GPU 0: NVIDIA RTX A4000
✓ CUDA tensor operation passed.
Result: [2. 4. 6.]


In [3]:
# ============================================================
# REINITIALIZE EXPERIMENT 3 PATHS AFTER KERNEL RESTART
# ============================================================

from pathlib import Path

EXPERIMENT_ROOT = Path(
    r"G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean"
)

DATASET_ROOT = Path(
    r"G:\Thesis_Hasan\Dataset\clean_dataset"
)

DATASET_YAML = (
    DATASET_ROOT /
    "data.yaml"
)

BEST_PT = (
    EXPERIMENT_ROOT /
    "train" /
    "weights" /
    "best.pt"
)

DEVICE = 0
IMG_SIZE = 640

print("=" * 90)
print("EXPERIMENT 3 PATH REINITIALIZATION")
print("=" * 90)

print("Experiment root:")
print(EXPERIMENT_ROOT)

print("\nDataset YAML:")
print(DATASET_YAML)

print("\nBest checkpoint:")
print(BEST_PT)

print("\nBEST.PT EXISTS:", BEST_PT.exists())
print("DATA YAML EXISTS:", DATASET_YAML.exists())

EXPERIMENT 3 PATH REINITIALIZATION
Experiment root:
G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean

Dataset YAML:
G:\Thesis_Hasan\Dataset\clean_dataset\data.yaml

Best checkpoint:
G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean\train\weights\best.pt

BEST.PT EXISTS: True
DATA YAML EXISTS: True


In [6]:
# ============================================================
# CHECKPOINT LOAD TEST
# ============================================================

import torch
from ultralytics import YOLO

print("=" * 90)
print("LOADING LFR BEST CHECKPOINT")
print("=" * 90)

if not BEST_PT.exists():
    raise FileNotFoundError(
        f"best.pt not found:\n{BEST_PT}"
    )

best_model = YOLO(
    str(BEST_PT)
)

print(
    "\n✓ best.pt loaded successfully."
)

best_model.info(
    detailed=False,
    verbose=True
)

LOADING LFR BEST CHECKPOINT

✓ best.pt loaded successfully.
YOLO12n_lfr summary: 276 layers, 2,505,173 parameters, 0 gradients, 7.0 GFLOPs


(276, 2505173, 0, 6.9515264)

In [7]:
# ============================================================
# CUDA FORWARD TEST
# ============================================================

device_obj = torch.device(
    f"cuda:{DEVICE}"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Using device:",
    device_obj
)

best_model.model.to(
    device_obj
)

dummy = torch.zeros(
    1,
    3,
    IMG_SIZE,
    IMG_SIZE,
    device=device_obj
)

print(
    "Running CUDA forward..."
)

with torch.no_grad():
    _ = best_model.model(dummy)

if torch.cuda.is_available():
    torch.cuda.synchronize()

print(
    "✓ LFR checkpoint CUDA forward passed."
)

Using device: cuda:0
Running CUDA forward...
✓ LFR checkpoint CUDA forward passed.


In [8]:
# CELL 10 — Validation

best_model = YOLO(str(BEST_PT))

val_metrics = best_model.val(
    data=str(DATASET_YAML),
    split="val",
    imgsz=IMG_SIZE,
    device=DEVICE,
    plots=True,
    project=str(EXPERIMENT_ROOT),
    name="validation",
    exist_ok=True,
    verbose=True,
)

print("=" * 90)
print("YOLO12n-LFR VALIDATION")
print("=" * 90)
print(f"Precision : {val_metrics.box.mp:.6f}")
print(f"Recall    : {val_metrics.box.mr:.6f}")
print(f"mAP50     : {val_metrics.box.map50:.6f}")
print(f"mAP50-95  : {val_metrics.box.map:.6f}")


Ultralytics 8.4.116  Python-3.13.5 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
YOLO12n_lfr summary (fused): 161 layers, 2,494,301 parameters, 0 gradients, 6.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 249.194.6 MB/s, size: 49.2 KB)
val: Scanning G:\Thesis_Hasan\Dataset\clean_dataset\valid\labels.cache... 231 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 231/231 21.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 3.1it/s 4.8s0.2s
                   all        231        360      0.701      0.613      0.659      0.412
  Cercospora leaf spot         34         96      0.591      0.375       0.43      0.179
          Flea-Beetles         37         62      0.597      0.484      0.529      0.284
          Healthy leaf         35         36      0.801      0.944      0.967      0.867
      Phomopsis Blight         28         33      0.728      0.697      0.736      0.398
   Phytophthora

In [11]:
# ============================================================
# RESTORE CLASS NAMES AFTER KERNEL RESTART
# ============================================================

CLASS_NAMES = [
    "Cercospora leaf spot",
    "Flea-Beetles",
    "Healthy leaf",
    "Phomopsis Blight",
    "Phytophthora Blight",
    "Powdery Mildew",
    "Tobacco Mosaic Virus",
]

assert len(CLASS_NAMES) == 7

print("✓ CLASS_NAMES restored")
print(CLASS_NAMES)

✓ CLASS_NAMES restored
['Cercospora leaf spot', 'Flea-Beetles', 'Healthy leaf', 'Phomopsis Blight', 'Phytophthora Blight', 'Powdery Mildew', 'Tobacco Mosaic Virus']


In [14]:
# ============================================================
# CELL 11 — Class-wise validation metrics
# Self-contained after kernel restart
# ============================================================

import json
import numpy as np
import pandas as pd

CLASS_NAMES = [
    "Cercospora leaf spot",
    "Flea-Beetles",
    "Healthy leaf",
    "Phomopsis Blight",
    "Phytophthora Blight",
    "Powdery Mildew",
    "Tobacco Mosaic Virus",
]

def extract_classwise_metrics(metrics, class_names):

    box = metrics.box

    class_indices = np.asarray(
        box.ap_class_index,
        dtype=int
    ).reshape(-1)

    precision = np.asarray(
        box.p,
        dtype=float
    ).reshape(-1)

    recall = np.asarray(
        box.r,
        dtype=float
    ).reshape(-1)

    ap50 = np.asarray(
        box.ap50,
        dtype=float
    ).reshape(-1)

    ap50_95 = np.asarray(
        box.ap,
        dtype=float
    ).reshape(-1)

    if not (
        len(class_indices)
        == len(precision)
        == len(recall)
        == len(ap50)
        == len(ap50_95)
    ):
        raise RuntimeError(
            "Class-wise metric arrays have inconsistent lengths."
        )

    rows = []

    for i, class_id in enumerate(class_indices):

        class_id = int(class_id)

        if not (
            0 <= class_id < len(class_names)
        ):
            raise ValueError(
                f"Invalid class ID: {class_id}"
            )

        rows.append({
            "class_id":
                class_id,

            "class_name":
                class_names[class_id],

            "precision":
                float(precision[i]),

            "recall":
                float(recall[i]),

            "mAP50":
                float(ap50[i]),

            "mAP50_95":
                float(ap50_95[i]),
        })

    full = pd.DataFrame({
        "class_id":
            np.arange(
                len(class_names)
            ),

        "class_name":
            class_names,
    })

    result = full.merge(
        pd.DataFrame(rows),
        on=[
            "class_id",
            "class_name",
        ],
        how="left"
    )

    return result


# ------------------------------------------------------------
# Extract validation class-wise metrics
# ------------------------------------------------------------

val_class_df = extract_classwise_metrics(
    val_metrics,
    CLASS_NAMES
)

print("=" * 90)
print("YOLO12n-LFR — VALIDATION CLASS-WISE METRICS")
print("=" * 90)

display(
    val_class_df
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

val_class_df.to_csv(
    EXPERIMENT_ROOT /
    "validation_per_class_metrics.csv",
    index=False
)


# ------------------------------------------------------------
# Overall validation metrics
# ------------------------------------------------------------

val_overall = {
    "precision":
        float(val_metrics.box.mp),

    "recall":
        float(val_metrics.box.mr),

    "mAP50":
        float(val_metrics.box.map50),

    "mAP50-95":
        float(val_metrics.box.map),
}

with open(
    EXPERIMENT_ROOT /
    "validation_metrics.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        val_overall,
        f,
        indent=2
    )


print(
    "\n✓ Validation class-wise metrics saved."
)

print(
    EXPERIMENT_ROOT /
    "validation_per_class_metrics.csv"
)

YOLO12n-LFR — VALIDATION CLASS-WISE METRICS


,class_id,class_name,precision,recall,mAP50,mAP50_95
0,0,Cercospora leaf spot,0.591234,0.375000,0.429624,0.179486
1,1,Flea-Beetles,0.597104,0.483871,0.528669,0.284020
2,2,Healthy leaf,0.800631,0.944444,0.967420,0.867042
3,3,Phomopsis Blight,0.728097,0.696970,0.736330,0.397882
4,4,Phytophthora Blight,0.848759,0.910714,0.939427,0.625089
5,5,Powdery Mildew,0.699958,0.507727,0.543957,0.315614
6,6,Tobacco Mosaic Virus,0.639032,0.370370,0.470678,0.214204



✓ Validation class-wise metrics saved.
G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean\validation_per_class_metrics.csv


In [15]:
# CELL 12 — Test evaluation

best_model = YOLO(str(BEST_PT))

test_metrics = best_model.val(
    data=str(DATASET_YAML),
    split="test",
    imgsz=IMG_SIZE,
    device=DEVICE,
    plots=True,
    project=str(EXPERIMENT_ROOT),
    name="test_evaluation",
    exist_ok=True,
    verbose=True,
)

print("=" * 90)
print("YOLO12n-LFR TEST")
print("=" * 90)
print(f"Precision : {test_metrics.box.mp:.6f}")
print(f"Recall    : {test_metrics.box.mr:.6f}")
print(f"mAP50     : {test_metrics.box.map50:.6f}")
print(f"mAP50-95  : {test_metrics.box.map:.6f}")


Ultralytics 8.4.116  Python-3.13.5 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
YOLO12n_lfr summary (fused): 161 layers, 2,494,301 parameters, 0 gradients, 6.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 374.1150.9 MB/s, size: 59.4 KB)
val: Scanning G:\Thesis_Hasan\Dataset\clean_dataset\test\labels.cache... 229 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 229/229 45.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 3.3it/s 4.5s0.2s
                   all        229        347      0.655      0.647      0.684      0.437
  Cercospora leaf spot         37        106      0.427      0.283      0.249     0.0768
          Flea-Beetles         41         61      0.572      0.557      0.596       0.31
          Healthy leaf         40         42      0.786      0.857      0.903      0.793
      Phomopsis Blight         27         31      0.748      0.863      0.874      0.598
   Phytophthora

In [16]:
# CELL 13 — Class-wise test metrics

test_class_df = extract_classwise_metrics(
    test_metrics,
    CLASS_NAMES
)

display(test_class_df)

test_class_df.to_csv(
    EXPERIMENT_ROOT / "per_class_test_metrics.csv",
    index=False
)

test_overall = {
    "precision": float(test_metrics.box.mp),
    "recall": float(test_metrics.box.mr),
    "mAP50": float(test_metrics.box.map50),
    "mAP50-95": float(test_metrics.box.map),
}

with open(
    EXPERIMENT_ROOT / "test_metrics.json",
    "w", encoding="utf-8"
) as f:
    json.dump(test_overall, f, indent=2)


,class_id,class_name,precision,recall,mAP50,mAP50_95
0,0,Cercospora leaf spot,0.427074,0.283019,0.249044,0.076756
1,1,Flea-Beetles,0.572487,0.557377,0.595833,0.310288
2,2,Healthy leaf,0.786450,0.857143,0.903086,0.792657
3,3,Phomopsis Blight,0.748153,0.862628,0.873989,0.598022
4,4,Phytophthora Blight,0.796527,0.900000,0.908374,0.611972
5,5,Powdery Mildew,0.675489,0.631579,0.761950,0.397933
6,6,Tobacco Mosaic Virus,0.581672,0.437500,0.495817,0.269962


In [17]:
# CELL 14 — Overall baseline comparison

BASELINE = {
    "precision": 0.607114098348289,
    "recall": 0.6943683279863577,
    "mAP50": 0.6878047677652831,
    "mAP50-95": 0.439391424683581,
}

rows = []

for metric, baseline_value in BASELINE.items():
    proposed_value = test_overall[metric]
    delta = proposed_value - baseline_value

    rows.append({
        "metric": metric,
        "YOLO12n_original": baseline_value,
        "YOLO12n_LFR": proposed_value,
        "delta": delta,
        "delta_percent": (
            100 * delta / baseline_value
            if baseline_value != 0
            else np.nan
        ),
    })

comparison_df = pd.DataFrame(rows)

display(comparison_df)

comparison_df.to_csv(
    EXPERIMENT_ROOT / "baseline_vs_lfr_comparison.csv",
    index=False
)


,metric,YOLO12n_original,YOLO12n_LFR,delta,delta_percent
0,precision,0.607114,0.655407,0.048293,7.954571
1,recall,0.694368,0.647035,-0.047333,-6.816740
2,mAP50,0.687805,0.684013,-0.003791,-0.551245
3,mAP50-95,0.439391,0.436799,-0.002593,-0.590088


In [18]:
# CELL 15 — Class-wise baseline vs LFR

BASELINE_CLASSWISE_CSV = Path(
    r"G:\Thesis_Hasan\Models\Baseline\baseline_analysis\bestpt_re_evaluation\bestpt_classwise_validation_test_metrics.csv"
)

if BASELINE_CLASSWISE_CSV.exists():

    baseline_all = pd.read_csv(BASELINE_CLASSWISE_CSV)

    baseline_test = baseline_all[
        (baseline_all["experiment_folder"] == "YOLOv12n_original")
        & (baseline_all["split"] == "test")
    ][[
        "class_name",
        "precision",
        "recall",
        "mAP50",
        "mAP50_95",
    ]].rename(columns={
        "precision": "baseline_precision",
        "recall": "baseline_recall",
        "mAP50": "baseline_mAP50",
        "mAP50_95": "baseline_mAP50_95",
    })

    lfr_test = test_class_df[[
        "class_name",
        "precision",
        "recall",
        "mAP50",
        "mAP50_95",
    ]].rename(columns={
        "precision": "lfr_precision",
        "recall": "lfr_recall",
        "mAP50": "lfr_mAP50",
        "mAP50_95": "lfr_mAP50_95",
    })

    class_comparison = baseline_test.merge(
        lfr_test,
        on="class_name",
        how="outer"
    )

    for metric in [
        "precision",
        "recall",
        "mAP50",
        "mAP50_95",
    ]:
        class_comparison[f"{metric}_delta"] = (
            class_comparison[f"lfr_{metric}"]
            - class_comparison[f"baseline_{metric}"]
        )

    display(class_comparison)

    class_comparison.to_csv(
        EXPERIMENT_ROOT / "baseline_vs_lfr_classwise_test.csv",
        index=False
    )

else:
    print("Baseline class-wise CSV not found:")
    print(BASELINE_CLASSWISE_CSV)


,class_name,baseline_precision,baseline_recall,baseline_mAP50,baseline_mAP50_95,lfr_precision,lfr_recall,lfr_mAP50,lfr_mAP50_95,precision_delta,recall_delta,mAP50_delta,mAP50_95_delta
0,Cercospora leaf spot,0.388843,0.390167,0.332700,0.116597,0.427074,0.283019,0.249044,0.076756,0.038232,-0.107148,-0.083656,-0.039840
1,Flea-Beetles,0.525228,0.590164,0.649241,0.312708,0.572487,0.557377,0.595833,0.310288,0.047258,-0.032787,-0.053408,-0.002420
2,Healthy leaf,0.663284,0.880952,0.886249,0.784368,0.786450,0.857143,0.903086,0.792657,0.123165,-0.023810,0.016837,0.008289
3,Phomopsis Blight,0.746022,0.853002,0.884627,0.598709,0.748153,0.862628,0.873989,0.598022,0.002131,0.009626,-0.010638,-0.000687
4,Phytophthora Blight,0.746457,0.925000,0.940440,0.631979,0.796527,0.900000,0.908374,0.611972,0.050070,-0.025000,-0.032066,-0.020007
5,Powdery Mildew,0.677212,0.736842,0.661465,0.381895,0.675489,0.631579,0.761950,0.397933,-0.001723,-0.105263,0.100485,0.016038
6,Tobacco Mosaic Virus,0.502622,0.484301,0.459994,0.249311,0.581672,0.437500,0.495817,0.269962,0.079051,-0.046801,0.035823,0.020651


In [19]:
# CELL 16 — Test inference + zero-prediction audit

TEST_IMAGES = DATASET_ROOT / "test" / "images"

inference_results = best_model.predict(
    source=str(TEST_IMAGES),
    imgsz=IMG_SIZE,
    conf=0.25,
    device=DEVICE,
    save=True,
    save_txt=True,
    project=str(EXPERIMENT_ROOT / "inference"),
    name="test_predictions_conf025",
    exist_ok=True,
    verbose=False,
)

zero_images = []

for result in inference_results:
    n_boxes = (
        0 if result.boxes is None
        else len(result.boxes)
    )

    if n_boxes == 0:
        zero_images.append(
            Path(result.path).name
        )

pd.DataFrame({
    "filename": zero_images
}).to_csv(
    EXPERIMENT_ROOT / "zero_prediction_images.csv",
    index=False
)

print("Test images:", len(inference_results))
print("Zero predictions:", len(zero_images))
print(
    f"Zero-prediction rate: "
    f"{100 * len(zero_images) / max(len(inference_results), 1):.2f}%"
)


Results saved to G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_Clean\inference\test_predictions_conf025
223 labels saved to G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_Clean\inference\test_predictions_conf025\labels
Test images: 229
Zero predictions: 6
Zero-prediction rate: 2.62%


In [21]:
# ============================================================
# CELL 17 — Confidence distribution
# Self-contained after kernel restart
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make sure the experiment path exists
from pathlib import Path

EXPERIMENT_ROOT = Path(
    r"G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean"
)

confidence_values = []

for result in inference_results:

    if (
        result.boxes is None
        or len(result.boxes) == 0
    ):
        continue

    confidence_values.extend(
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
        .tolist()
    )

confidence_values = np.asarray(
    confidence_values,
    dtype=float
)

print("=" * 90)
print("YOLO12n-LFR — TEST PREDICTION CONFIDENCE")
print("=" * 90)

if len(confidence_values):

    # --------------------------------------------------------
    # Save raw confidence values
    # --------------------------------------------------------

    confidence_df = pd.DataFrame({
        "confidence":
            confidence_values
    })

    confidence_csv = (
        EXPERIMENT_ROOT /
        "test_prediction_confidences.csv"
    )

    confidence_df.to_csv(
        confidence_csv,
        index=False
    )

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    print(
        "Prediction count:",
        len(confidence_values)
    )

    print(
        f"Mean confidence  : "
        f"{confidence_values.mean():.6f}"
    )

    print(
        f"Median confidence: "
        f"{np.median(confidence_values):.6f}"
    )

    print(
        f"Std deviation    : "
        f"{confidence_values.std():.6f}"
    )

    print(
        f"Minimum          : "
        f"{confidence_values.min():.6f}"
    )

    print(
        f"Maximum          : "
        f"{confidence_values.max():.6f}"
    )

    # --------------------------------------------------------
    # Confidence histogram
    # --------------------------------------------------------

    plt.figure(
        figsize=(9, 5)
    )

    plt.hist(
        confidence_values,
        bins=20
    )

    plt.xlabel(
        "Prediction confidence"
    )

    plt.ylabel(
        "Prediction count"
    )

    plt.title(
        "YOLO12n-LFR Test Prediction Confidence Distribution"
    )

    plt.tight_layout()

    histogram_path = (
        EXPERIMENT_ROOT /
        "test_prediction_confidence_histogram.png"
    )

    plt.savefig(
        histogram_path,
        dpi=220,
        bbox_inches="tight"
    )

    plt.show()

    print(
        "\nSaved confidence CSV:"
    )

    print(
        confidence_csv
    )

    print(
        "\nSaved histogram:"
    )

    print(
        histogram_path
    )

else:

    print(
        "No predictions found."
    )

YOLO12n-LFR — TEST PREDICTION CONFIDENCE
Prediction count: 374
Mean confidence  : 0.607977
Median confidence: 0.600320
Std deviation    : 0.230976
Minimum          : 0.250992
Maximum          : 0.975215


<Figure size 900x500 with 1 Axes>


Saved confidence CSV:
G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean\test_prediction_confidences.csv

Saved histogram:
G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean\test_prediction_confidence_histogram.png


In [22]:
# CELL 18 — Complexity and speed

complexity = {}

try:
    complexity["parameters"] = int(
        sum(
            p.numel()
            for p in best_model.model.parameters()
        )
    )
except Exception as e:
    complexity["parameter_error"] = repr(e)

try:
    complexity["model_info"] = str(
        best_model.info(
            detailed=False,
            verbose=False
        )
    )
except Exception as e:
    complexity["model_info_error"] = repr(e)

try:
    sample_images = list(
        TEST_IMAGES.glob("*.jpg")
    )[:50]

    if sample_images:

        best_model.predict(
            source=[str(p) for p in sample_images[:4]],
            imgsz=IMG_SIZE,
            device=DEVICE,
            verbose=False
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        t0 = time.perf_counter()

        best_model.predict(
            source=[str(p) for p in sample_images],
            imgsz=IMG_SIZE,
            device=DEVICE,
            verbose=False
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - t0

        complexity["timed_images"] = len(sample_images)
        complexity["wallclock_latency_ms_per_image"] = (
            1000 * elapsed / len(sample_images)
        )
        complexity["approx_fps"] = (
            len(sample_images) / elapsed
        )

        print(
            f"Latency: "
            f"{complexity['wallclock_latency_ms_per_image']:.3f} ms/image"
        )
        print(
            f"Approx FPS: "
            f"{complexity['approx_fps']:.2f}"
        )

except Exception as e:
    complexity["speed_error"] = repr(e)

with open(
    EXPERIMENT_ROOT / "model_complexity_speed.json",
    "w", encoding="utf-8"
) as f:
    json.dump(complexity, f, indent=2, default=str)

print("Saved:", EXPERIMENT_ROOT / "model_complexity_speed.json")


Results saved to G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_Clean\inference\test_predictions_conf025
226 labels saved to G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_Clean\inference\test_predictions_conf025\labels
Saved: G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean\model_complexity_speed.json


In [26]:
# CELL 19 — Experiment manifest

manifest = {
    "experiment": "YOLO12n_LFR_clean",
    "phase": "proposed_model_ablation_3",
    "base_model": "YOLO12n",
    "dataset_condition": "clean",
    "dataset_yaml": str(DATASET_YAML),
    "imgsz": IMG_SIZE,
    "device": DEVICE,
    "seed": SEED,
    "epochs": EPOCHS,
    "batch": BATCH,
    "patience": PATIENCE,
    "architectural_change": (
        "Added one lightweight local feature refinement block "
        "to the existing P3 path immediately before Detect: "
        "DWConv 3x3 followed by Conv 1x1. P4 and P5 unchanged."
    ),
    "p2_branch": False,
    "lcsa": False,
    "previous_ablation_results": {
        "YOLO12n_P2": "rejected",
        "YOLO12n_LCSA": "rejected",
    },
    "baseline_reference": {
        "test_precision": 0.607114098348289,
        "test_recall": 0.6943683279863577,
        "test_mAP50": 0.6878047677652831,
        "test_mAP50_95": 0.439391424683581,
    },
}

with open(
    EXPERIMENT_ROOT / "experiment_manifest.json",
    "w", encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=2)

print("Experiment manifest saved.")


Experiment manifest saved.


# Experiment 3 decision gate

Primary:
- Test mAP50-95
- Cercospora mAP50-95
- Cercospora recall

Secondary:
- Overall mAP50
- precision/recall balance
- TMV performance
- confusion matrix
- PR/F1 curves

Efficiency:
- parameter count
- GFLOPs
- latency
- FPS

If LFR improves the frozen baseline and helps the difficult classes, it becomes a candidate component. If it fails, reject it independently and return to the original YOLO12n baseline for the next ablation.

Do not combine P2, LCSA, and LFR merely to force a gain.


In [27]:
# CELL 20 — Final experiment summary

print("=" * 90)
print("YOLO12n-LFR EXPERIMENT 3 SUMMARY")
print("=" * 90)

print("Experiment:", "YOLO12n_LFR_clean")
print("Best PT:", BEST_PT)
print("Dataset:", DATASET_YAML)
print("Image size:", IMG_SIZE)

if "test_overall" in globals():
    print("\nTEST RESULTS")
    for key, value in test_overall.items():
        print(f"{key:12s}: {value:.6f}")

if "comparison_df" in globals():
    print("\nBASELINE COMPARISON")
    display(comparison_df)

print("\nExperiment directory:")
print(EXPERIMENT_ROOT)
print("=" * 90)


YOLO12n-LFR EXPERIMENT 3 SUMMARY
Experiment: YOLO12n_LFR_clean
Best PT: G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean\train\weights\best.pt
Dataset: G:\Thesis_Hasan\Dataset\clean_dataset\data.yaml
Image size: 640

TEST RESULTS
precision   : 0.655407
recall      : 0.647035
mAP50       : 0.684013
mAP50-95    : 0.436799

BASELINE COMPARISON


,metric,YOLO12n_original,YOLO12n_LFR,delta,delta_percent
0,precision,0.607114,0.655407,0.048293,7.954571
1,recall,0.694368,0.647035,-0.047333,-6.816740
2,mAP50,0.687805,0.684013,-0.003791,-0.551245
3,mAP50-95,0.439391,0.436799,-0.002593,-0.590088



Experiment directory:
G:\Thesis_Hasan\Models\Experiment\exp_3\YOLO12n_LFR_clean
